# Measurement Landscapes

Measurement landscapes attach a scalar measurement to the cycle representative along each barcode bar. Use them when bar birth and death are not enough and you want features such as length, area, curvature, circularity, or component counts.

This notebook shows the core workflow: choose a `cycle_func`, compute landscapes on a shared grid, plot them, and turn them into vectors.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from persforest import PersistenceForest
from persforest.cycle_rep_vectorisations import (
    constant_one_functional,
    signed_chain_area,
    signed_chain_circularity,
    signed_chain_connected_components,
    signed_chain_edge_length,
    signed_chain_excess_curvature,
    signed_chain_non_circularity,
    signed_chain_volume,
)


def sample_noisy_star(points_per_edge=14, inner_radius=0.48, outer_radius=1.2, noise=0.02, seed=11):
    rng = np.random.default_rng(seed)
    angles = np.linspace(np.pi / 2, np.pi / 2 + 2.0 * np.pi, 10, endpoint=False)
    radii = np.where(np.arange(10) % 2 == 0, outer_radius, inner_radius)
    vertices = np.column_stack((radii * np.cos(angles), radii * np.sin(angles)))

    edge_points = []
    for start, end in zip(vertices, np.roll(vertices, -1, axis=0)):
        t = np.linspace(0.0, 1.0, points_per_edge, endpoint=False)[:, None]
        edge_points.append((1.0 - t) * start + t * end)

    points = np.vstack(edge_points)
    return points + rng.normal(scale=noise, size=points.shape)


def sample_noisy_sphere(n=90, noise=0.04, seed=8):
    rng = np.random.default_rng(seed)
    z = rng.uniform(-1.0, 1.0, n)
    theta = rng.uniform(0.0, 2.0 * np.pi, n)
    radius = 1.0 + rng.normal(scale=noise, size=n)
    xy = np.sqrt(1.0 - z * z)
    sphere = np.column_stack((xy * np.cos(theta), xy * np.sin(theta), z))
    return radius[:, None] * sphere


## Existing cycle functionals

A `cycle_func` is any callable with signature `cycle_func(chain, point_cloud) -> float`. The ready-made scalar functionals in `persforest.cycle_rep_vectorisations` include:

- General: `constant_one_functional`, `signed_chain_volume`
- Planar 1-cycle geometry: `signed_chain_edge_length`, `signed_chain_area`, `signed_chain_excess_curvature`, `signed_chain_excess_curvature_normalized`, `signed_chain_excess_curvature_diff_to_unsigned`, `signed_chain_circularity`, `signed_chain_circularity_complement`, `signed_chain_non_circularity`
- Component and tendril diagnostics: `signed_chain_connected_components`, `signed_chain_excess_connected_components`, `signed_chain_connected_components_only_signed_simplices`, `signed_chain_avg_tendril_length`, `signed_chain_num_of_branching_points`, `signed_chain_num_of_branching_points_only_signed_simplices`, `signed_chain_tendril_branching_ratio`

Most geometry and component functionals are for planar 1-cycles. `signed_chain_volume` is dimension-independent.


## Build a star-shaped example

A circle is useful for debugging, but it hides what geometric measurements are doing. This noisy star-shaped boundary has one dominant loop and non-convex geometry, so length, area, curvature, and circularity lead to visibly different landscapes.


In [ ]:
points = sample_noisy_star()
forest = PersistenceForest(points)

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(points[:, 0], points[:, 1], s=9, color="black")
ax.set_aspect("equal")
ax.set_title("Noisy star-shaped point cloud")
plt.show()

forest.plot_barcode(min_bar_length=0.01, coloring="forest")


## Compute landscape families

Each call stores a family in `forest.landscape_families[label]` when `cache=True`. Use the same `x_grid` when you want feature vectors that are directly comparable.


In [ ]:
grid = np.linspace(0.0, 1.4, 120)

landscape_specs = [
    ("standard", constant_one_functional),
    ("edge length", signed_chain_edge_length),
    ("area", signed_chain_area),
    ("excess curvature", signed_chain_excess_curvature),
    ("circularity", signed_chain_circularity),
    ("non-circularity", signed_chain_non_circularity),
]

for label, cycle_func in landscape_specs:
    forest.compute_measurement_landscapes(
        cycle_func=cycle_func,
        label=label,
        max_k=3,
        x_grid=grid,
        min_bar_length=0.05,
        cache=True,
        cache_functionals=True,
    )

sorted(forest.landscape_families)


## Plot landscape families

`ks` selects which landscape levels to plot. If omitted, all computed levels are drawn.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

forest.plot_measurement_landscapes(
    label="edge length",
    ks=[1, 2, 3],
    ax=axes[0],
    title="Edge-length landscapes",
    linewidth=2.0,
    show=False,
)

forest.plot_measurement_landscapes(
    label="excess curvature",
    ks=[1, 2, 3],
    ax=axes[1],
    title="Excess-curvature landscapes",
    linewidth=2.0,
    show=False,
)

plt.tight_layout()
plt.show()


## Vectorize landscapes

`evaluate_on_grid` returns a NumPy array with shape `(levels, len(grid))`. Flatten it when your downstream method expects one feature vector per forest.


In [ ]:
geometry_family = forest.landscape_families["excess curvature"]
values = geometry_family.evaluate_on_grid(grid, levels=3)
feature_vector = values.reshape(-1)

values.shape, feature_vector.shape


## Reuse cached barcode functionals

With `cache_functionals=True`, per-bar scalar measurements are available in `forest.barcode_functionals[label]`. This is useful when you want to inspect the raw measurements before or after building landscapes.


In [ ]:
barcode_functionals = forest.barcode_functionals["non-circularity"]
bar_values = barcode_functionals.evaluate_on_grid(grid)

type(barcode_functionals), len(barcode_functionals.bars), bar_values.shape


## Signed versus unsigned evaluation

By default, measurement landscapes evaluate functionals after cancelling opposite-oriented duplicate simplices. Pass `signed=True` when the functional should see the stored signed chain.

The small circle-with-spoke example below produces representatives with some opposite-oriented duplicate edges. Edge length changes when those duplicates are preserved.


In [ ]:
theta = np.linspace(0.0, 2.0 * np.pi, 50, endpoint=False)
circle = np.column_stack((np.cos(theta), np.sin(theta)))
spoke = np.column_stack((np.linspace(0.0, 0.9, 12), np.zeros(12)))
spoke_points = np.vstack((circle, spoke))
spoke_forest = PersistenceForest(spoke_points)

fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(spoke_points[:, 0], spoke_points[:, 1], s=10, color="black")
ax.set_aspect("equal")
ax.set_title("Circle with a spoke")
plt.show()

spoke_grid = np.linspace(0.0, 1.1, 120)
for label, signed in [("signed length", True), ("unsigned length", False)]:
    spoke_forest.compute_measurement_landscapes(
        cycle_func=signed_chain_edge_length,
        label=label,
        max_k=2,
        x_grid=spoke_grid,
        min_bar_length=0.05,
        signed=signed,
        cache=True,
    )

spoke_forest.plot_landscape_comparison_between_functionals(
    labels=["signed length", "unsigned length"],
    k=1,
    title="Signed and unsigned edge-length landscapes",
)
plt.show()


## A 3D landscape example

For a 3D point cloud, cycle representatives are surface chains. `signed_chain_volume` sums simplex volumes in arbitrary dimension and can be used as a simple 3D functional.


In [ ]:
points_3d = sample_noisy_sphere()
forest_3d = PersistenceForest(points_3d)

grid_3d = np.linspace(0.0, 2.0, 100)
boundary_family = forest_3d.compute_measurement_landscapes(
    cycle_func=signed_chain_volume,
    label="boundary simplex volume",
    max_k=2,
    x_grid=grid_3d,
    min_bar_length=0.1,
    cache=True,
)

forest_3d.plot_measurement_landscapes(
    label="boundary simplex volume",
    title="3D boundary simplex-volume landscapes",
)

boundary_family.evaluate_on_grid(grid_3d, levels=2).shape
